In [3]:
from pathlib import Path
import pandas as pd
import h3
import json

In [ ]:
def clean_rides(rides, df):
    df['started_at'] = pd.to_datetime(df['started_at'])
    df['ended_at'] = pd.to_datetime(df['ended_at'])
    df = df.dropna(subset=['start_lat', 'start_lng', 'end_lat', 'end_lng'])
    df = df[df['ended_at'] > df['started_at']]
    duration = (df['ended_at'] - df['started_at']).dt.total_seconds()
    df = df[(duration >= 60) & (duration <= 60*120)]

    df['minutes'] = df['started_at'].dt.hour * 60 + df['started_at'].dt.minute
    breaks = [-1, 360, 600, 960, 1200, 1439]
    labels = ['early_morning', 'morning', 'midday', 'evening', 'night']
    df['time'] = pd.cut(df['minutes'], bins=breaks, labels=labels, right=False)
    df['month'] = df['started_at'].dt.month
    df['weekend'] = df['started_at'].dt.dayofweek >= 5

    df['start_cell'] = [h3.latlng_to_cell(lat, lng, 9) for lat, lng in zip(df['start_lat'], df['start_lng'])]
    df['end_cell'] = [h3.latlng_to_cell(lat, lng, 9) for lat, lng in zip(df['start_lat'], df['start_lng'])]

    new_rides = df.groupby(['time', 'month', 'weekend', 'start_cell', 'end_cell']).size().reset_index(name='ride_count')
    
    return pd.concat([rides, new_rides], ignore_index=True)

In [ ]:
citibike_data = Path("../Data/citibike")

rides = pd.DataFrame()

for file in citibike_data.rglob('*.csv'):
    df = pd.read_csv(file)
    rides = clean_rides(rides, df)

/var/folders/vw/hn8f6x7d3558mwg78cw3__hr0000gn/T/ipykernel_3459/2119734977.py:6: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/vw/hn8f6x7d3558mwg78cw3__hr0000gn/T/ipykernel_3459/2119734977.py:6: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/vw/hn8f6x7d3558mwg78cw3__hr0000gn/T/ipykernel_3459/2119734977.py:6: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/vw/hn8f6x7d3558mwg78cw3__hr0000gn/T/ipykernel_3459/2119734977.py:6: DtypeWarning: Columns (0: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/vw/hn8f6x7d3558mwg78cw3__hr0000gn/T/ipykernel_3459/2119734977.py:6: DtypeWarning: Columns (0: end_s

In [ ]:
cells = pd.DataFrame(pd.concat([rides['start_cell'], rides['end_cell']]).drop_duplicates(), columns = ["station_cells"])
cells = cells['station_cells']

features = []
for hex in cells:
    vertices = h3.cell_to_boundary(hex)
    border = [[lng, lat] for lat, lng in vertices]
    border.append(border[0])
    lat, lng = h3.cell_to_latlng(hex)

    features.append({
        "type": "Feature",
        "properties": {
            "h3_index": hex,
            "center_lat": lat,
            "center_lng": lng,
        },
        "geometry": {
            "type": "Polygon",
            "coordinates": [border],
        },
    })

cells = {
    "type": "FeatureCollection",
    "features": features,
}

In [ ]:
rides.to_csv('../Intermediate/01_citibike_rides.csv', index=False)
with open("../Intermediate/02_citibike_cells.geojson", "w") as f:
    json.dump(cells, f)

In [3]:
import pandas as pd
import numpy as np
from shapely import wkt
import geopandas as gpd                                                 

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RES = 9

In [53]:
schools = pd.read_csv("../Data/nyc_open_data/2019_-_2020_School_Locations_20260727.csv", low_memory=False)
schools = schools[schools["Status_descriptions"].str.contains("Open", case=False, na=False)]
schools_geo = schools.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()
schools_geo['h3_cell'] = [
    h3.latlng_to_cell(lat, lon, RES)
    for lat, lon in zip(schools_geo['LATITUDE'], schools_geo['LONGITUDE'])
]
schools_geo = schools_geo[['Location_Category_Description', 'h3_cell']].rename(columns = {'Location_Category_Description':'school_type'})
schools_geo['has_school'] = 1

mapping = {   
    'Early Childhood': 'early_schooling',                                                                                          
    'Elementary': 'elementary_school',
    'High School': 'high_school',   
    'Junior High-Intermediate-Middle':'middle_school',                             
    'K-8': 'k_8_school',                                                                              
    'K-12 all grades': 'k_12_school',   
    'Seconary School': 'secondary_school',                                                                       
    'Ungraded': 'ungraded_school'                                                                        
}                                                                                                                                                                                                                        
schools_geo['school_type'] = schools_geo['school_type'].replace(mapping)

schools_geo = schools_geo.pivot_table(                                                                     
    index='h3_cell',                                                                                         
    columns='school_type',                                                                                   
    values='has_school',                                                                                     
    aggfunc='max',                                                
    fill_value=0                                                                                             
).reset_index()

In [54]:
bikes = pd.read_csv('../Data/nyc_open_data/New_York_City_Bike_Routes_20260727.csv', low_memory=False)
bikes = bikes[bikes["status"].str.contains("Current", case=False, na=False)]

def line_to_cells(wkt_str, res=RES):
    geom = wkt.loads(wkt_str)
    lines = geom.geoms if geom.geom_type == 'MultiLineString' else [geom]
    cells = set()
    for line in lines:
        coords = list(line.coords)  # (lon, lat)
        vertex_cells = [h3.latlng_to_cell(lat, lon, res) for lon, lat in coords]
        cells.update(vertex_cells)
        for a, b in zip(vertex_cells[:-1], vertex_cells[1:]):
            if a != b:
                cells.update(h3.grid_path_cells(a, b))
    return cells

bikes['h3_cells'] = bikes['the_geom'].apply(line_to_cells)
bikes['on_street'] = np.where(bikes['onoffst'] == "ON", 1, 0)
bikes['protected_lane'] = np.where((bikes['ft_facilit'] == "Protected") | (bikes['tf_facilit'] == "Protected"), 1, 0)
bikes = bikes[['on_street', 'protected_lane', 'h3_cells']]
bikes = (bikes.explode('h3_cells') 
         .rename(columns={'h3_cells': 'h3_cell'})  
         .drop_duplicates(subset=['h3_cell']))

In [55]:
parks = pd.read_csv('../Data/nyc_open_data/Parks_Zones_20260727.csv', low_memory=False)
parks = parks[parks["RETIRED"] == False]

def polygon_to_cells(wkt_str, res=RES):
    if isinstance(wkt_str, str): 
        geom = wkt.loads(wkt_str)
    else:
        geom = wkt_str
    polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
    cells = set()
    for poly in polys:
        outer = [(lat, lon) for lon, lat in poly.exterior.coords]
        holes = [[(lat, lon) for lon, lat in interior.coords] for interior in poly.interiors]
        cells.update(h3.h3shape_to_cells(h3.LatLngPoly(outer, *holes), res))
    if not cells:
        c = geom.centroid
        cells.add(h3.latlng_to_cell(c.y, c.x, res))
    return cells

parks['h3_cells'] = parks['multipolygon'].apply(polygon_to_cells)
parks['has_park'] = 1
parks = parks.rename(columns = {'PROPNAME':'park_name'})
parks = parks[['has_park', 'park_name', 'h3_cells']]
parks = (parks.explode('h3_cells')  
            .rename(columns={'h3_cells': 'h3_cell'})  
            .drop_duplicates(subset=['h3_cell'])[['h3_cell', 'has_park', 'park_name']])

In [ ]:
import geopandas as gpd                                                                                      
import pandas as pd                                                                                          
import h3                                                                                                    
from shapely.geometry import Polygon                                                                         
                                                                                                                
RES = 9                                                                                                      
                                                                                                                
pluto = gpd.read_file("../Data/nyc_open_data/nyc_mappluto_26v1_shp.zip")[['BBL', 'geometry']]                
pluto['BBL'] = pluto['BBL'].astype(int)                                                                                                                                                                           
pluto = pluto.to_crs(epsg=4326)                                                                              
                                                                                                                
zoning = pd.read_csv('../Data/nyc_open_data/NYC_Zoning_Tax_Lot_Database_20260727.csv', dtype=str)            
zoning['BBL'] = zoning['BBL'].astype(int)                                                                    
zoning = zoning.drop_duplicates()                                                                            
                                                                                                                
zone_cols = ['Zoning District 1', 'Zoning District 2', 'Zoning District 3', 'Zoning District 4']             
zoning['res_zoning_ct'] = 0                                                                                  
zoning['com_zoning_ct'] = 0                                                                                  
zoning['manf_zoning_ct'] = 0                                                                                 
                                                                                                                
for col in zone_cols:                                                                                        
    if col in zoning.columns:                                                                                
        first_char = zoning[col].fillna('').astype(str).str.strip().str.upper().str[0]                       
        zoning['res_zoning_ct'] += (first_char == 'R').astype(int)                                           
        zoning['com_zoning_ct'] += (first_char == 'C').astype(int)                                           
        zoning['manf_zoning_ct'] += (first_char == 'M').astype(int)                                          
                                                                                                                
zoning = zoning[['BBL', 'res_zoning_ct', 'com_zoning_ct', 'manf_zoning_ct']].drop_duplicates()                                                                                                                     
zoning_gdf = pluto.merge(zoning, on="BBL", how="inner")                                                      
                                                                                                                
xmin, ymin, xmax, ymax = zoning_gdf.total_bounds                                                             
nyc_bbox = Polygon([(xmin, ymin), (xmin, ymax), (xmax, ymax), (xmax, ymin)])                                                                                                                                           
nyc_outer = [(lat, lon) for lon, lat in nyc_bbox.exterior.coords]                                            
nyc_h3_cells = h3.h3shape_to_cells(h3.LatLngPoly(nyc_outer), RES)                                            
                                                                                                                
h3_polys = []                                                                                                
for cell in nyc_h3_cells:                                                                                    
    boundary = h3.cell_to_boundary(cell)  # [(lat, lon), ...]                                                
    poly_geom = Polygon([(lon, lat) for lat, lon in boundary])                                               
    h3_polys.append({'h3_cell': cell, 'geometry': poly_geom})                                                                                                                                                             
h3_gdf = gpd.GeoDataFrame(h3_polys, crs="EPSG:4326")                                                                                                          
joined = gpd.sjoin(zoning_gdf, h3_gdf, how="inner", predicate="intersects")                                  
                                                           
zoning_h3 = joined.groupby('h3_cell').agg(                                                           
    res_lots=('res_zoning_ct', 'sum'),                                                                       
    com_lots=('com_zoning_ct', 'sum'),                                                                       
    manf_lots=('manf_zoning_ct', 'sum')                                                                      
).reset_index()                                                                                              
                                                                                                                
zoning_h3

Loading MapPLUTO shapes...
Reprojecting MapPLUTO to EPSG:4326 (Lat/Lon)...
Processing zoning counts...
Building NYC H3 Grid...
Created NYC H3 Grid with 20,647 hexagon cells.
Performing Vectorized Spatial Join (gpd.sjoin)...
Aggregating zoning counts per H3 cell...
DONE! Total H3 cells with zoning: 8,271


,h3_cell,res_lots,com_lots,manf_lots
0,892a100000bffff,1,0,0
1,892a100001bffff,1,0,0
2,892a100002bffff,66,0,2
3,892a100002fffff,11,0,7
4,892a1000043ffff,0,0,0


In [2]:
import pandas as pd
import numpy as np
from shapely import wkt
from shapely.geometry import Polygon                                                
import geopandas as gpd   
import h3                                                                                                    

RES = 9

In [43]:
station_cells = pd.read_csv('../Intermediate/01_citibike_rides.csv').rename(
    columns = {'start_cell':'cell_start', 'end_cell':'cell_end'})
emp = pd.read_csv('../Intermediate/03_employment_data.csv').rename(columns = {'h3_index':'cell'})
census = pd.read_csv('../Intermediate/04_census_block_data.csv').rename(columns = {'h3_index':'cell'})
schools = pd.read_csv('../Intermediate/05_school_data.csv').rename(columns = {'h3_cell':'cell'})
bike_lanes = pd.read_csv('../Intermediate/06_bike_lane_data.csv').rename(columns = {'h3_cell':'cell'})
parks = pd.read_csv('../Intermediate/07_parks_data.csv').rename(columns = {'h3_cell':'cell'})
zoning = pd.read_csv('../Intermediate/08_zoning_data.csv').rename(columns = {'h3_cell':'cell'})

In [44]:
combined = station_cells.copy()

def merge_tgt(df):
    tmp = combined.merge(df.add_suffix('_start'), how = "left", on= "cell_start")
    return tmp.merge(df.add_suffix('_end'), how = "left", on= "cell_end")

combined = merge_tgt(emp)
combined = merge_tgt(census)
combined = merge_tgt(schools)
combined = merge_tgt(bike_lanes)
combined = merge_tgt(parks)
combined = merge_tgt(zoning)

In [48]:
combined["grid_distance"] = combined.apply(
    lambda row: h3.grid_distance(row["cell_start"], row["cell_end"]),
    axis=1,
)

In [52]:
station_cells

,time,month,weekend,cell_start,cell_end,ride_count
0,early_morning,9,False,892a1001203ffff,892a1001203ffff,4
1,early_morning,9,False,892a100120bffff,892a100120bffff,1
2,early_morning,9,False,892a100120fffff,892a100120fffff,3
3,early_morning,9,False,892a1001213ffff,892a1001213ffff,1
4,early_morning,9,False,892a1001217ffff,892a1001217ffff,1
...,...,...,...,...,...,...
714677,night,5,True,892a1077693ffff,892a1077693ffff,3
714678,night,5,True,892a1077697ffff,892a1077697ffff,4
714679,night,5,True,892a107769bffff,892a107769bffff,1
714680,night,5,True,892a10776d3ffff,892a10776d3ffff,47


In [ ]:
pd.read_csv("../Intermediate/09_combined_dataset.csv")

In [1]:
import pandas as pd
import h3

#----------------------------------
# Import Cleaned Data
#----------------------------------

station_cells = pd.read_csv('../Intermediate/01_citibike_rides.csv').rename(
    columns = {'start_cell':'cell_start', 'end_cell':'cell_end'})
emp = pd.read_csv('../Intermediate/03_employment_data.csv').rename(columns = {'h3_index':'cell'})
census = pd.read_csv('../Intermediate/04_census_block_data.csv').rename(columns = {'h3_index':'cell'})
schools = pd.read_csv('../Intermediate/05_school_data.csv').rename(columns = {'h3_cell':'cell'})
bike_lanes = pd.read_csv('../Intermediate/06_bike_lane_data.csv').rename(columns = {'h3_cell':'cell'})
parks = pd.read_csv('../Intermediate/07_parks_data.csv').rename(columns = {'h3_cell':'cell'})
zoning = pd.read_csv('../Intermediate/08_zoning_data.csv').rename(columns = {'h3_cell':'cell'})

In [2]:
combined = station_cells.copy()

def merge_tgt(df):
    tmp = combined.merge(df.add_suffix('_start'), how = "left", on= "cell_start")
    return tmp.merge(df.add_suffix('_end'), how = "left", on= "cell_end")

In [3]:
combined = merge_tgt(emp)

In [4]:
combined = merge_tgt(census)

In [5]:
combined = merge_tgt(schools)

In [6]:
combined = merge_tgt(bike_lanes)

In [7]:
combined = merge_tgt(parks)

In [8]:
combined = merge_tgt(zoning)

In [ ]:
combined["grid_distance"] = combined.apply(
    lambda row: h3.grid_distance(row["cell_start"], row["cell_end"]),
    axis=1,
)

In [ ]:
#import polars as pl

#combined_pl = pl.from_pandas(combined)

combined_pl.write_csv('../Intermediate/09_combined_dataset.csv')

In [18]:
combined_pl

time,month,weekend,cell_start,cell_end,ride_count,all_day_jobs_start,sex_female_start,pay_1250_or_less_start,job_ct_start,pay_over_3333_start,pay_1251_to_3333_start,white_collar_jobs_start,entertainment_jobs_start,age_29_or_younger_start,early_start_jobs_start,all_day_jobs_end,sex_female_end,pay_1250_or_less_end,job_ct_end,pay_over_3333_end,pay_1251_to_3333_end,white_collar_jobs_end,entertainment_jobs_end,age_29_or_younger_end,early_start_jobs_end,housing_units_start,age_15_to_34_start,total_pop_start,female_pop_start,housing_units_end,age_15_to_34_end,total_pop_end,female_pop_end,High school_start,Secondary School_start,early_schooling_start,elementary_school_start,k_12_school_start,k_8_school_start,middle_school_start,ungraded_school_start,High school_end,Secondary School_end,early_schooling_end,elementary_school_end,k_12_school_end,k_8_school_end,middle_school_end,ungraded_school_end,on_street_start,protected_lane_start,on_street_end,protected_lane_end,has_park_start,park_name_start,has_park_end,park_name_end,res_lots_start,com_lots_start,manf_lots_start,res_lots_end,com_lots_end,manf_lots_end
str,i64,bool,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,f64,f64,f64,f64,f64,f64
"""early_morning""",9,false,"""892a1001203ffff""","""892a1001213ffff""",1,0.253927,0.479058,0.17801,76.4,0.569808,0.252182,0.352531,0.185864,0.196335,0.207679,0.364839,0.434783,0.060491,48.090909,0.691871,0.247637,0.043478,0.20983,0.179584,0.381853,128.84,0.294713,296.56,0.520232,153.65,0.280457,380.45,0.524379,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,0.0,1.0,0.0,null,null,null,null,75.0,20.0,1.0,67.0,13.0,0.0
"""early_morning""",9,false,"""892a1001203ffff""","""892a100a88bffff""",1,0.253927,0.479058,0.17801,76.4,0.569808,0.252182,0.352531,0.185864,0.196335,0.207679,0.088773,0.563969,0.052219,42.555556,0.718016,0.229765,0.010444,0.143603,0.127937,0.75718,128.84,0.294713,296.56,0.520232,148.84,0.306293,420.12,0.542226,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,0.0,1.0,0.0,null,null,null,null,75.0,20.0,1.0,80.0,0.0,0.0
"""early_morning""",9,false,"""892a1001203ffff""","""892a100ac87ffff""",1,0.253927,0.479058,0.17801,76.4,0.569808,0.252182,0.352531,0.185864,0.196335,0.207679,0.09141,0.503855,0.090859,113.5,0.718062,0.191079,0.247797,0.154736,0.178414,0.506057,128.84,0.294713,296.56,0.520232,69.826087,0.330548,202.956522,0.524207,null,null,null,null,null,null,null,null,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,null,null,null,null,75.0,20.0,1.0,109.0,11.0,21.0
"""early_morning""",9,false,"""892a1001203ffff""","""892a100ac8bffff""",1,0.253927,0.479058,0.17801,76.4,0.569808,0.252182,0.352531,0.185864,0.196335,0.207679,0.556193,0.685498,0.194562,220.666667,0.451662,0.353776,0.165861,0.25136,0.157704,0.026586,128.84,0.294713,296.56,0.520232,137.65,0.317589,413.9,0.492752,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1.0,0.0,1.0,0.0,null,null,null,null,75.0,20.0,1.0,50.0,55.0,0.0
"""early_morning""",9,false,"""892a100120bffff""","""892a1001207ffff""",1,0.002613,0.532811,0.199768,1148.0,0.589141,0.211092,0.141405,0.031069,0.157085,0.824913,0.150424,0.563559,0.055085,36.307692,0.758475,0.186441,0.243644,0.110169,0.108051,0.495763,0.0,0.0,0.0,0.0,224.666667,0.281634,536.866667,0.535204,null,null,null,null,null,null,null,null,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,"""Harris Park""",null,null,5.0,1.0,2.0,167.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""night""",5,true,"""892a10776d7ffff""","""892a1072cc3ffff""",1,0.58973,0.425405,0.077838,115.625,0.585405,0.336757,0.043784,0.194054,0.115135,0.172432,0.077642,0.457677,0.174548,171.3,0.592528,0.232925,0.151781,0.701401,